In [36]:
# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np

In [37]:
# ============================================================
# 2. LOAD DATA
# ============================================================

matches = pd.read_csv("matches.csv")
team_match_stats = pd.read_csv("team_match_stats.csv")
player_match_stats = pd.read_csv("player_match_stats_with_positions.csv")
positions = pd.read_csv("positions.csv")
team_master = pd.read_csv("team_master.csv")
player_master = pd.read_csv("player_master.csv")

print(matches.shape)
print(team_match_stats.shape)
print(player_match_stats.shape)
print(positions.shape)

(171, 17)
(342, 19)
(5336, 28)
(8492, 15)


In [38]:
# ============================================================
# 3. CREATE MATCH CONTEXT
# ============================================================

home_context = matches[
    [
        "match_id",
        "match_date",
        "match_label",
        "home_team",
        "away_team",
        "home_score",
        "away_score",
        "home_result"
    ]
].rename(
    columns={
        "home_team": "team",
        "away_team": "opponent",
        "home_score": "goals_for",
        "away_score": "goals_against",
        "home_result": "result"
    }
)

away_context = matches[
    [
        "match_id",
        "match_date",
        "match_label",
        "away_team",
        "home_team",
        "away_score",
        "home_score",
        "away_result"
    ]
].rename(
    columns={
        "away_team": "team",
        "home_team": "opponent",
        "away_score": "goals_for",
        "home_score": "goals_against",
        "away_result": "result"
    }
)

match_context = pd.concat(
    [home_context, away_context],
    ignore_index=True
)

match_context.head()

,match_id,match_date,match_label,team,opponent,goals_for,goals_against,result
0,3922240,2024-01-28,2024-01-28 | Egypt 1-1 Congo DR,Egypt,Congo DR,1,1,Draw
1,3920400,2024-01-19,2024-01-19 | Senegal 3-1 Cameroon,Senegal,Cameroon,3,1,Win
2,3920395,2024-01-17,2024-01-17 | Congo DR 1-1 Zambia,Congo DR,Zambia,1,1,Draw
3,3920407,2024-01-21,2024-01-21 | South Africa 4-0 Namibia,South Africa,Namibia,4,0,Win
4,3920412,2024-01-23,2024-01-23 | Guinea 0-2 Senegal,Guinea,Senegal,0,2,Loss


In [39]:
# ============================================================
# 4. CREATE HIGHLIGHTED TEAM MATCHES
# ============================================================

highlighted_team_matches = team_match_stats.merge(
    match_context,
    on=["match_id", "team"],
    how="left"
)

highlighted_team_matches["match_score"] = (
    highlighted_team_matches["xg"] * 0.35 +
    highlighted_team_matches["shots"] * 0.20 +
    highlighted_team_matches["passes"] * 0.10 +
    highlighted_team_matches["pressures"] * 0.15 +
    highlighted_team_matches["recoveries"] * 0.10 +
    highlighted_team_matches["pass_completion_pct"] * 0.10
)

highlighted_team_matches["match_score_rank"] = (
    highlighted_team_matches
    .groupby(["tournament_id", "team"])["match_score"]
    .rank(method="first", ascending=False)
)

highlighted_team_matches = highlighted_team_matches[
    highlighted_team_matches["match_score_rank"] <= 5
].copy()

highlighted_team_matches.head()

,competition_name,season_name,competition_short,tournament_id,match_id,team,total_events,shots,passes,carries,...,completed_passes,pass_completion_pct,match_date,match_label,opponent,goals_for,goals_against,result,match_score,match_score_rank
0,African Cup of Nations,2023,AFCON,AFCON_2023,3920384,Côte d'Ivoire,1998,10,598,510,...,517,86.454849,2024-01-13,2024-01-13 | Côte d'Ivoire 2-0 Guinea-Bissau,Guinea-Bissau,2,0,Win,90.942899,2.0
1,African Cup of Nations,2023,AFCON,AFCON_2023,3920384,Guinea-Bissau,1450,10,394,301,...,328,83.248731,2024-01-13,2024-01-13 | Côte d'Ivoire 2-0 Guinea-Bissau,Côte d'Ivoire,0,2,Loss,75.260238,1.0
2,African Cup of Nations,2023,AFCON,AFCON_2023,3920385,Egypt,2025,22,589,497,...,512,86.926995,2024-01-14,2024-01-14 | Egypt 2-2 Mozambique,Mozambique,2,2,Draw,99.717919,1.0
3,African Cup of Nations,2023,AFCON,AFCON_2023,3920385,Mozambique,1445,6,373,329,...,309,82.841823,2024-01-14,2024-01-14 | Egypt 2-2 Mozambique,Egypt,2,2,Draw,71.139052,2.0
4,African Cup of Nations,2023,AFCON,AFCON_2023,3920387,Cape Verde Islands,1511,15,417,320,...,341,81.774580,2024-01-14,2024-01-14 | Ghana 1-2 Cape Verde Islands,Ghana,2,1,Win,75.529533,3.0


In [40]:
# ============================================================
# 5. ADD POSITION CONTEXT TO PLAYER MATCH STATS
# ============================================================

player_context_cols = [
    "tournament_id",
    "team",
    "player",
    "position",
    "position_group",
    "general_role",
    "offensive_role",
    "defensive_role",
    "importance_score",
    "role_adjusted_importance_score",
    "importance_tier",
    "strengths",
    "weaknesses",
    "best_role_fits",
    "best_role_fit_scores"
]

existing_player_context_cols = [
    c for c in player_context_cols
    if c in player_master.columns
]

player_match_stats_enriched = player_match_stats.merge(
    player_master[existing_player_context_cols],
    on=["tournament_id", "team", "player"],
    how="left",
    suffixes=("", "_master")
)

if "position_group" not in player_match_stats_enriched.columns:
    player_match_stats_enriched["position_group"] = np.where(
        player_match_stats_enriched["position"].astype(str).eq("Goalkeeper"),
        "Goalkeeper",
        "Outfield"
    )

player_match_stats_enriched.head()

,competition_name,season_name,competition_short,tournament_id,match_id,team,player,total_events,shots,passes,...,position_group,general_role,offensive_role,defensive_role,importance_score,importance_tier,strengths,weaknesses,best_role_fits,best_role_fit_scores
0,African Cup of Nations,2023,AFCON,AFCON_2023,3920384,Côte d'Ivoire,Franck Yannick Kessié,134,1,39,...,Midfielder,Box Threat,Finisher / Shot Threat,Ball Winner,79.38,Important Player,"['Strong xG generation', 'High shot volume', '...",['No clear major weakness from selected indica...,"Box Threat, Ball Winner, Progressive Carrier","86.6, 83.8, 81.9"
1,African Cup of Nations,2023,AFCON,AFCON_2023,3920384,Côte d'Ivoire,Ghislain Konan,230,0,72,...,Defender,Tempo Controller,High-Volume Passer,Ball Winner,78.18,Important Player,"['High passing involvement', 'High carry volum...",['No clear major weakness from selected indica...,"Ball Winner, Progressive Carrier, Tempo Contro...","94.3, 94.3, 84.3"
2,African Cup of Nations,2023,AFCON,AFCON_2023,3920384,Côte d'Ivoire,Ibrahim Sangaré,210,0,66,...,Midfielder,Pressing Player,Balanced Offensive Role,Pressing Specialist,70.91,Important Player,['High pressing activity'],['No clear major weakness from selected indica...,"Pressing Player, Progressive Carrier, Tempo Co...","86.3, 74.7, 71.1"
3,African Cup of Nations,2023,AFCON,AFCON_2023,3920384,Côte d'Ivoire,Idrissa Doumbia,11,0,4,...,Midfielder,Balanced Role,Balanced Offensive Role,Balanced Defensive Role,9.00,Low Usage / Limited Impact,['Balanced profile without one dominant streng...,"['Low passing involvement', 'Low carry volume'...","Box Threat, Tempo Controller, Ball Winner","20.0, 18.5, 11.8"
4,African Cup of Nations,2023,AFCON,AFCON_2023,3920384,Côte d'Ivoire,Jean-Philippe Krasso,83,4,10,...,Forward,Balanced Role,Balanced Offensive Role,Balanced Defensive Role,37.37,Low Usage / Limited Impact,['Balanced profile without one dominant streng...,['Low passing involvement'],"Box Threat, Balanced Role, Pressing Player","76.9, 38.8, 37.2"


In [41]:
# ============================================================
# 6. CREATE HIGHLIGHTED PLAYER MATCHES
# ============================================================

highlighted_player_matches = player_match_stats_enriched.merge(
    match_context,
    on=["match_id", "team"],
    how="left"
)

for col in [
    "xg",
    "shots",
    "passes",
    "pressures",
    "recoveries",
    "duels",
    "interceptions",
    "blocks",
    "clearances",
    "pass_completion_pct"
]:
    if col not in highlighted_player_matches.columns:
        highlighted_player_matches[col] = 0

highlighted_player_matches["base_match_score"] = (
    highlighted_player_matches["xg"] * 0.35 +
    highlighted_player_matches["shots"] * 0.20 +
    highlighted_player_matches["passes"] * 0.10 +
    highlighted_player_matches["pressures"] * 0.15 +
    highlighted_player_matches["recoveries"] * 0.10 +
    highlighted_player_matches["pass_completion_pct"] * 0.10
)

highlighted_player_matches["goalkeeper_match_score"] = (
    highlighted_player_matches["passes"] * 0.12 +
    highlighted_player_matches["pass_completion_pct"] * 0.12 +
    highlighted_player_matches["recoveries"] * 0.12 +
    highlighted_player_matches["duels"] * 0.12 +
    highlighted_player_matches["blocks"] * 0.18 +
    highlighted_player_matches["clearances"] * 0.20 +
    highlighted_player_matches["total_events"] * 0.14
)

highlighted_player_matches["match_score"] = np.where(
    highlighted_player_matches["position_group"].astype(str).eq("Goalkeeper"),
    highlighted_player_matches["goalkeeper_match_score"],
    highlighted_player_matches["base_match_score"]
)

highlighted_player_matches["match_score_rank"] = (
    highlighted_player_matches
    .groupby(["tournament_id", "team", "player"])["match_score"]
    .rank(method="first", ascending=False)
)

highlighted_player_matches = highlighted_player_matches[
    highlighted_player_matches["match_score_rank"] <= 5
].copy()

highlighted_player_matches.head()

,competition_name,season_name,competition_short,tournament_id,match_id,team,player,total_events,shots,passes,...,match_date,match_label,opponent,goals_for,goals_against,result,base_match_score,goalkeeper_match_score,match_score,match_score_rank
0,African Cup of Nations,2023,AFCON,AFCON_2023,3920384,Côte d'Ivoire,Franck Yannick Kessié,134,1,39,...,2024-01-13,2024-01-13 | Côte d'Ivoire 2-0 Guinea-Bissau,Guinea-Bissau,2,0,Win,15.199140,35.612308,15.199140,2.0
1,African Cup of Nations,2023,AFCON,AFCON_2023,3920384,Côte d'Ivoire,Ghislain Konan,230,0,72,...,2024-01-13,2024-01-13 | Côte d'Ivoire 2-0 Guinea-Bissau,Guinea-Bissau,2,0,Win,17.694444,52.373333,17.694444,3.0
2,African Cup of Nations,2023,AFCON,AFCON_2023,3920384,Côte d'Ivoire,Ibrahim Sangaré,210,0,66,...,2024-01-13,2024-01-13 | Côte d'Ivoire 2-0 Guinea-Bissau,Guinea-Bissau,2,0,Win,19.236364,48.583636,19.236364,1.0
3,African Cup of Nations,2023,AFCON,AFCON_2023,3920384,Côte d'Ivoire,Idrissa Doumbia,11,0,4,...,2024-01-13,2024-01-13 | Côte d'Ivoire 2-0 Guinea-Bissau,Guinea-Bissau,2,0,Win,8.000000,11.140000,8.000000,1.0
4,African Cup of Nations,2023,AFCON,AFCON_2023,3920384,Côte d'Ivoire,Jean-Philippe Krasso,83,4,10,...,2024-01-13,2024-01-13 | Côte d'Ivoire 2-0 Guinea-Bissau,Guinea-Bissau,2,0,Win,8.444815,19.180000,8.444815,3.0


In [42]:
# ============================================================
# 7. POSITION ORDER
# ============================================================

position_order = {
    "Goalkeeper": 1,
    "Right Back": 2,
    "Right Center Back": 3,
    "Center Back": 4,
    "Left Center Back": 5,
    "Left Back": 6,
    "Right Wing Back": 7,
    "Left Wing Back": 8,
    "Right Defensive Midfield": 9,
    "Center Defensive Midfield": 10,
    "Left Defensive Midfield": 11,
    "Right Center Midfield": 12,
    "Center Midfield": 13,
    "Left Center Midfield": 14,
    "Right Midfield": 15,
    "Left Midfield": 16,
    "Right Attacking Midfield": 17,
    "Center Attacking Midfield": 18,
    "Left Attacking Midfield": 19,
    "Right Wing": 20,
    "Left Wing": 21,
    "Right Center Forward": 22,
    "Center Forward": 23,
    "Left Center Forward": 24,
    "Secondary Striker": 25
}

In [43]:
# ============================================================
# 8. CREATE MOST FREQUENT XI BASE
# ============================================================

starter_positions = positions[
    positions["starter"] == True
].copy()

most_frequent_base = (
    starter_positions
    .groupby([
        "competition_name",
        "season_name",
        "competition_short",
        "tournament_id",
        "team",
        "player",
        "position",
        "position_x",
        "position_y"
    ])
    .agg(
        starts=("match_id", "nunique"),
        jersey_number=("jersey_number", lambda x: x.dropna().mode()[0] if len(x.dropna()) > 0 else np.nan)
    )
    .reset_index()
)

most_frequent_base["position_order"] = (
    most_frequent_base["position"]
    .map(position_order)
    .fillna(99)
)

most_frequent_base.head()

,competition_name,season_name,competition_short,tournament_id,team,player,position,position_x,position_y,starts,jersey_number,position_order
0,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Adam Ounas,Right Wing,88.0,68.0,1,12,20
1,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Anthony Louis Mandréa,Goalkeeper,8.0,40.0,3,16,1
2,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Aïssa Mandi,Right Center Back,25.0,52.0,3,2,3
3,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Baghdad Bounedjah,Center Forward,105.0,40.0,3,9,23
4,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Fares Chaïbi,Right Center Midfield,60.0,52.0,1,17,12


In [44]:
# ============================================================
# 9. ADD PLAYER MASTER INFO TO XI BASE
# ============================================================

xi_extra_cols = [
    "tournament_id",
    "team",
    "player",
    "position_group",
    "general_role",
    "offensive_role",
    "defensive_role",
    "importance_score",
    "role_adjusted_importance_score",
    "importance_tier",
    "strengths",
    "weaknesses",
    "best_role_fits",
    "best_role_fit_scores"
]

existing_xi_extra_cols = [
    c for c in xi_extra_cols
    if c in player_master.columns
]

most_frequent_base = most_frequent_base.merge(
    player_master[existing_xi_extra_cols],
    on=["tournament_id", "team", "player"],
    how="left"
)

if "role_adjusted_importance_score" not in most_frequent_base.columns:
    most_frequent_base["role_adjusted_importance_score"] = most_frequent_base["importance_score"]

if "position_group" not in most_frequent_base.columns:
    most_frequent_base["position_group"] = np.where(
        most_frequent_base["position"].astype(str).eq("Goalkeeper"),
        "Goalkeeper",
        "Outfield"
    )

most_frequent_base.head()

,competition_name,season_name,competition_short,tournament_id,team,player,position,position_x,position_y,starts,...,general_role,offensive_role,defensive_role,importance_score,importance_tier,strengths,weaknesses,best_role_fits,best_role_fit_scores,role_adjusted_importance_score
0,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Adam Ounas,Right Wing,88.0,68.0,1,...,Balanced Role,Balanced Offensive Role,Balanced Defensive Role,37.68,Low Usage / Limited Impact,['Balanced profile without one dominant streng...,"['Low pressing activity', 'Limited ball-winnin...","Box Threat, Progressive Carrier, Balanced Role","52.9, 52.4, 34.1",37.68
1,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Anthony Louis Mandréa,Goalkeeper,8.0,40.0,3,...,Goalkeeper,Build-Up Goalkeeper,Goal Prevention,47.74,Low Usage / Limited Impact,['Active in recoveries'],"['Low clearance activity', 'Limited shot-stopp...","Tempo Controller, Ball Winner, Progressive Car...","59.3, 49.5, 38.5",47.74
2,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Aïssa Mandi,Right Center Back,25.0,52.0,3,...,Tempo Controller,High-Volume Passer,Balanced Defensive Role,75.95,Important Player,"['High passing involvement', 'Strong passing a...",['Low pressing activity'],"Progressive Carrier, Tempo Controller, Box Threat","99.3, 97.3, 75.4",75.95
3,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Baghdad Bounedjah,Center Forward,105.0,40.0,3,...,Box Threat,Finisher / Shot Threat,Balanced Defensive Role,53.12,Rotation / Contributor,"['Strong xG generation', 'High shot volume']",['No clear major weakness from selected indica...,"Box Threat, Pressing Player, Balanced Role","98.2, 79.2, 54.5",53.12
4,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Fares Chaïbi,Right Center Midfield,60.0,52.0,1,...,Box Threat,Finisher / Shot Threat,Balanced Defensive Role,60.57,Rotation / Contributor,['High shot volume'],['Limited ball-winning activity'],"Box Threat, Progressive Carrier, Tempo Controller","77.2, 72.0, 61.6",60.57


In [45]:
# ============================================================
# 10. CREATE MOST FREQUENT XI — UNIQUE POSITIONS
# ============================================================

selected_rows = []

for (tournament_id, team), group in most_frequent_base.groupby(["tournament_id", "team"]):

    group = group.sort_values(
        ["starts", "role_adjusted_importance_score", "importance_score"],
        ascending=[False, False, False]
    ).copy()

    used_positions = set()
    used_players = set()
    team_rows = []

    # First pass: one player per position
    for _, row in group.iterrows():

        if (
            row["position"] not in used_positions and
            row["player"] not in used_players
        ):
            team_rows.append(row)
            used_positions.add(row["position"])
            used_players.add(row["player"])

        if len(team_rows) == 11:
            break

    # Second pass: if fewer than 11, complete with best remaining players
    if len(team_rows) < 11:

        for _, row in group.iterrows():

            if row["player"] not in used_players:
                team_rows.append(row)
                used_players.add(row["player"])

            if len(team_rows) == 11:
                break

    team_xi = pd.DataFrame(team_rows)

    if len(team_xi) > 0:
        team_xi = team_xi.sort_values("position_order").copy()
        team_xi["xi_number"] = range(1, len(team_xi) + 1)
        selected_rows.append(team_xi)

most_frequent_xi = pd.concat(
    selected_rows,
    ignore_index=True
)

most_frequent_xi = most_frequent_xi.sort_values(
    ["tournament_id", "team", "xi_number"]
)

most_frequent_xi.head(20)

,competition_name,season_name,competition_short,tournament_id,team,player,position,position_x,position_y,starts,...,offensive_role,defensive_role,importance_score,importance_tier,strengths,weaknesses,best_role_fits,best_role_fit_scores,role_adjusted_importance_score,xi_number
0,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Anthony Louis Mandréa,Goalkeeper,8.0,40.0,3,...,Build-Up Goalkeeper,Goal Prevention,47.74,Low Usage / Limited Impact,['Active in recoveries'],"['Low clearance activity', 'Limited shot-stopp...","Tempo Controller, Ball Winner, Progressive Car...","59.3, 49.5, 38.5",47.74,1
1,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Youcef Atal,Right Back,28.0,68.0,3,...,High-Volume Passer,Balanced Defensive Role,73.25,Important Player,"['High passing involvement', 'High carry volume']",['No clear major weakness from selected indica...,"Progressive Carrier, Tempo Controller, Ball Wi...","86.0, 82.8, 77.6",73.25,2
2,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Aïssa Mandi,Right Center Back,25.0,52.0,3,...,High-Volume Passer,Balanced Defensive Role,75.95,Important Player,"['High passing involvement', 'Strong passing a...",['Low pressing activity'],"Progressive Carrier, Tempo Controller, Box Threat","99.3, 97.3, 75.4",75.95,3
3,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Ramy Bensebaini,Left Center Back,25.0,28.0,2,...,High-Volume Passer,Ball Winner,72.86,Important Player,"['High passing involvement', 'Strong passing a...",['No clear major weakness from selected indica...,"Progressive Carrier, Tempo Controller, Ball Wi...","98.5, 93.7, 87.3",72.86,4
4,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Rayan Aït Nouri,Left Back,28.0,12.0,3,...,High-Volume Passer,Ball Winner,87.85,Key Player,"['High passing involvement', 'High carry volum...",['No clear major weakness from selected indica...,"Progressive Carrier, Tempo Controller, Ball Wi...","99.6, 91.2, 86.2",87.85,5
5,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Ramiz Larbi Zerrouki,Center Defensive Midfield,45.0,40.0,1,...,High-Volume Passer,Ball Winner,72.57,Important Player,"['High passing involvement', 'High carry volum...",['No clear major weakness from selected indica...,"Progressive Carrier, Tempo Controller, Ball Wi...","93.6, 86.0, 78.3",72.57,6
6,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Hichem Boudaoui,Right Center Midfield,60.0,52.0,1,...,Finisher / Shot Threat,Balanced Defensive Role,67.92,Important Player,"['High shot volume', 'Strong passing accuracy']",['No clear major weakness from selected indica...,"Box Threat, Progressive Carrier, Tempo Controller","90.5, 77.6, 72.3",67.92,7
7,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Ismael Bennacer,Left Center Midfield,60.0,28.0,1,...,High-Volume Passer,Balanced Defensive Role,72.47,Important Player,"['High passing involvement', 'High carry volume']",['No clear major weakness from selected indica...,"Progressive Carrier, Tempo Controller, Balance...","86.8, 75.3, 65.4",72.47,8
8,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Riyad Mahrez,Right Wing,88.0,68.0,2,...,Balanced Offensive Role,Balanced Defensive Role,52.59,Rotation / Contributor,['Balanced profile without one dominant streng...,['No clear major weakness from selected indica...,"Ball Winner, Progressive Carrier, Tempo Contro...","68.6, 63.4, 55.3",52.59,9
9,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Mohamed Youcef Belaïli,Left Wing,88.0,12.0,2,...,Finisher / Shot Threat,Balanced Defensive Role,76.89,Important Player,"['High shot volume', 'High passing involvement...",['No clear major weakness from selected indica...,"Box Threat, Progressive Carrier, Balanced Role","91.8, 86.8, 71.6",76.89,10


In [46]:
# ============================================================
# 11. CREATE XI DISPLAY COLUMNS
# ============================================================

most_frequent_xi["xi_label"] = (
    most_frequent_xi["xi_number"].astype(str) +
    ". " +
    most_frequent_xi["player"].astype(str) +
    " | " +
    most_frequent_xi["position"].astype(str)
)

most_frequent_xi["xi_role_label"] = (
    most_frequent_xi["general_role"].fillna("N/A").astype(str) +
    " | " +
    most_frequent_xi["importance_tier"].fillna("N/A").astype(str)
)

most_frequent_xi["starts_share"] = (
    most_frequent_xi["starts"] /
    most_frequent_xi.groupby(["tournament_id", "team"])["starts"].transform("max") *
    100
)

most_frequent_xi.head()

,competition_name,season_name,competition_short,tournament_id,team,player,position,position_x,position_y,starts,...,importance_tier,strengths,weaknesses,best_role_fits,best_role_fit_scores,role_adjusted_importance_score,xi_number,xi_label,xi_role_label,starts_share
0,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Anthony Louis Mandréa,Goalkeeper,8.0,40.0,3,...,Low Usage / Limited Impact,['Active in recoveries'],"['Low clearance activity', 'Limited shot-stopp...","Tempo Controller, Ball Winner, Progressive Car...","59.3, 49.5, 38.5",47.74,1,1. Anthony Louis Mandréa | Goalkeeper,Goalkeeper | Low Usage / Limited Impact,100.000000
1,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Youcef Atal,Right Back,28.0,68.0,3,...,Important Player,"['High passing involvement', 'High carry volume']",['No clear major weakness from selected indica...,"Progressive Carrier, Tempo Controller, Ball Wi...","86.0, 82.8, 77.6",73.25,2,2. Youcef Atal | Right Back,Tempo Controller | Important Player,100.000000
2,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Aïssa Mandi,Right Center Back,25.0,52.0,3,...,Important Player,"['High passing involvement', 'Strong passing a...",['Low pressing activity'],"Progressive Carrier, Tempo Controller, Box Threat","99.3, 97.3, 75.4",75.95,3,3. Aïssa Mandi | Right Center Back,Tempo Controller | Important Player,100.000000
3,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Ramy Bensebaini,Left Center Back,25.0,28.0,2,...,Important Player,"['High passing involvement', 'Strong passing a...",['No clear major weakness from selected indica...,"Progressive Carrier, Tempo Controller, Ball Wi...","98.5, 93.7, 87.3",72.86,4,4. Ramy Bensebaini | Left Center Back,Tempo Controller | Important Player,66.666667
4,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Rayan Aït Nouri,Left Back,28.0,12.0,3,...,Key Player,"['High passing involvement', 'High carry volum...",['No clear major weakness from selected indica...,"Progressive Carrier, Tempo Controller, Ball Wi...","99.6, 91.2, 86.2",87.85,5,5. Rayan Aït Nouri | Left Back,Tempo Controller | Key Player,100.000000


In [47]:
# ============================================================
# 12. ROUND AND SAVE OUTPUTS
# ============================================================

for df in [highlighted_team_matches, highlighted_player_matches, most_frequent_xi]:
    numeric_cols = df.select_dtypes(include=np.number).columns
    df[numeric_cols] = df[numeric_cols].round(2)

highlighted_matches = pd.concat(
    [
        highlighted_team_matches.assign(level="team"),
        highlighted_player_matches.assign(level="player")
    ],
    ignore_index=True
)

highlighted_matches.to_csv(
    "highlighted_matches.csv",
    index=False
)

most_frequent_xi.to_csv(
    "most_frequent_xi.csv",
    index=False
)

print("highlighted_matches.csv saved successfully!")
print("most_frequent_xi.csv saved successfully!")

print("highlighted_matches:", highlighted_matches.shape)
print("most_frequent_xi:", most_frequent_xi.shape)

highlighted_matches.csv saved successfully!
most_frequent_xi.csv saved successfully!
highlighted_matches: (5446, 50)
most_frequent_xi: (1056, 27)
